# Selective prediction under simulated image shift

Five independently generated datasets each contain 3,200 training images and 1,200 held-out 16 × 16 images. Two standardized logistic models see clean data or paired clean and corrupted data. Blur, gain and noise perturb the test images. Each model accepts the most confident 80% using distance from probability 0.5.

Published result: Under simulated shift, augmentation reduced error at 80% coverage from 31.0% to 26.4%. On clean images, that error increased from 10.6% to 13.2%. Results describe the simulated signal and corruption family.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'selective-medical-image-shift-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", 'numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2'], check=True)

### train.py

In [ ]:
(root / 'train.py').write_text('"""Matched-coverage selective classification under simulated acquisition shift."""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.ndimage import gaussian_filter\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\n\ndef images(n, seed):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(2, size=n)\n    row, col = np.mgrid[-1:1:16j, -1:1:16j]\n    x = rng.normal(0, 0.26, (n, 16, 16))\n    x = gaussian_filter(x, (0, 0.65, 0.65))\n    centers = rng.normal(0, 0.24, (n, 2, 1, 1))\n    radius = (row - centers[:, 0]) ** 2 + (col - centers[:, 1]) ** 2\n    x += (0.3 + 0.23 * y[:, None, None]) * np.exp(-radius / 0.16)\n    return x, y\n\n\ndef corrupt(x, seed):\n    rng = np.random.default_rng(seed)\n    blur = gaussian_filter(x, (0, 1.1, 1.1))\n    return blur * rng.uniform(0.35, 1.25, (len(x), 1, 1)) + rng.normal(0, 0.16, x.shape)\n\n\ndef evaluate(y, p):\n    error = (p >= 0.5) != y\n    order = np.argsort(-np.abs(p - 0.5), kind="stable")\n    risks = np.cumsum(error[order]) / np.arange(1, len(y) + 1)\n    return {\n        "auroc": roc_auc_score(y, p),\n        "aurc": risks.mean(),\n        "risk_at_80_percent": risks[int(0.8 * len(y)) - 1],\n    }\n\n\ndef run(seed):\n    x, y = images(3200, seed)\n    test, target = images(1200, seed + 100)\n    records = {}\n    for method, features, labels in (\n        ("baseline", x, y),\n        ("augmented", np.concatenate([x, corrupt(x, seed + 200)]), np.tile(y, 2)),\n    ):\n        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))\n        model.fit(features.reshape(len(features), -1), labels)\n        dest = Path(__file__).with_name("checkpoints")\n        dest.mkdir(exist_ok=True)\n        scaler, lr = model.steps[0][1], model.steps[1][1]\n        np.savez_compressed(\n            dest / f"{method}-{seed}.npz",\n            mean=scaler.mean_,\n            scale=scaler.scale_,\n            coef=lr.coef_,\n            intercept=lr.intercept_,\n        )\n        for name, inputs in (("clean", test), ("shifted", corrupt(test, seed + 300))):\n            records[f"{method}_{name}"] = evaluate(\n                target, model.predict_proba(inputs.reshape(len(inputs), -1))[:, 1]\n            )\n    return records\n\n\nif __name__ == "__main__":\n    seeds = [17, 29, 43, 59, 71]\n    runs = {seed: run(seed) for seed in seeds}\n    sample = runs[seeds[0]]\n    means = {\n        key: {\n            metric: np.mean([run[key][metric] for run in runs.values()])\n            for metric in sample[key]\n        }\n        for key in sample\n    }\n    report = {"data": "synthetic 16x16 images", "runs": runs, "means": means}\n    Path(__file__).with_name("results.json").write_text(json.dumps(report, indent=2))\n    print(json.dumps(means, indent=2))\n', encoding="utf8")

### Train and evaluate

In [ ]:
subprocess.run([sys.executable, 'train.py'], cwd=root, check=True)

In [ ]:
import json
report = json.loads((root / "results.json").read_text())
print(json.dumps(report.get("means", report.get("subject_macro")), indent=2))